# Generate 48 episode images with Stable Diffusion XL
Paste the complete `image-prompts.md` document you received from Telegram into the box below. Run the cells from top to bottom. This notebook generates the 48 images one at a time, saves each completed image, and packages them as a ZIP accepted by the repository's 48-image importer.

**No Python editing is needed.** Choose a Colab GPU runtime first: Runtime → Change runtime type → T4 GPU (if offered). Free GPU availability and session limits are controlled by Google. The first run downloads the public SDXL model and can take a while.

**Important:** This notebook does not need a Gemini key, image API key, Cloudflare, or GitHub token. It creates a ZIP for you to download; you then upload that ZIP to the repository and run the existing Telegram ZIP Import workflow.

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors pillow ipywidgets

In [ ]:
import torch, os, re, io, zipfile, shutil, time
from pathlib import Path
from PIL import Image
from IPython.display import display, Markdown
import ipywidgets as widgets

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab choose Runtime → Change runtime type → T4 GPU (if available), then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
WORK = Path('/content/science-channel-image-job')
IMAGE_DIR = WORK / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Working folder:', WORK)

## 1. Paste the Telegram prompt document
Copy the entire contents of `image-prompts.md` from Telegram and paste it into this box. Keep the `## Scene 01` through `## Scene 48` headings.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
prompt_box = widgets.Textarea(
    value='', placeholder='Paste the complete image-prompts.md text here…',
    description='Prompts:', layout=widgets.Layout(width='100%', height='360px'),
    style={'description_width':'initial'}
)
display(prompt_box)

In [ ]:
def parse_scene_prompts(markdown_text):
    matches = list(re.finditer(r'(?m)^##\s*Scene\s*(\d+)\s*$', markdown_text))
    prompts = {}
    for index, match in enumerate(matches):
        number = int(match.group(1))
        end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown_text)
        body = markdown_text[match.end():end].strip()
        body = re.sub(r'\n{3,}', '\n\n', body)
        if not body:
            raise ValueError(f'Scene {number:02d} has no prompt text.')
        if number in prompts:
            raise ValueError(f'Duplicate scene heading: {number:02d}')
        prompts[number] = body
    expected = set(range(1, 49))
    if set(prompts) != expected:
        missing = sorted(expected - set(prompts))
        extra = sorted(set(prompts) - expected)
        raise ValueError(f'Expected scenes 01–48. Missing: {missing}; unexpected: {extra}. Paste the full image-prompts.md file.')
    return [prompts[n] for n in range(1, 49)]

PROMPTS = parse_scene_prompts(prompt_box.value)
print(f'Validated {len(PROMPTS)} numbered prompts.')
print('Scene 01 preview:', PROMPTS[0][:350] + ('…' if len(PROMPTS[0]) > 350 else ''))

## 2. Load Stable Diffusion XL
This loads the public SDXL Base 1.0 model. It uses half precision and GPU memory offloading to fit common Colab GPUs.

In [ ]:
from diffusers import StableDiffusionXLPipeline, EulerAncestralDiscreteScheduler
model_id = 'stabilityai/stable-diffusion-xl-base-1.0'
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, variant='fp16', use_safetensors=True
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()
pipe.enable_model_cpu_offload()
print('SDXL is loaded. You can now generate the 48 images.')

## 3. Generate all 48 images
Click the cell's Run button once. Images are saved after each scene. If Colab disconnects, reconnect, rerun the setup/load cells, paste the same prompts, and rerun this cell; already completed scene files are skipped.

In [ ]:
from tqdm.auto import tqdm
negative_prompt = 'text, letters, words, watermark, logo, signature, blurry, low quality, low detail, distorted anatomy, extra limbs, duplicate subjects, flat lighting, oversaturated, jpeg artifacts'
for scene_number, prompt in enumerate(tqdm(PROMPTS, desc='Generating scenes'), start=1):
    target = IMAGE_DIR / f'scene-{scene_number:02d}.webp'
    if target.exists() and target.stat().st_size > 1000:
        continue
    generator = torch.Generator(device='cpu').manual_seed(42000 + scene_number)
    result = pipe(
        prompt=prompt + '\n\nPremium cinematic 3D CGI science documentary still, polished physically based materials, realistic global illumination, rich fine detail, strong composition, coherent environment, wide 16:9 movie frame, no text.',
        negative_prompt=negative_prompt,
        width=1024, height=576, num_inference_steps=28, guidance_scale=6.5,
        generator=generator
    )
    image = result.images[0].convert('RGB')
    image.save(target, 'WEBP', quality=88, method=6)
    del result, image
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f'Saved scene {scene_number:02d}/48')
print('Image generation loop finished.')

## 4. Validate and download the 48-image ZIP
The ZIP uses numbered WebP filenames and is designed for the repository's existing 48-image ZIP importer. The importer has a 25 MiB ZIP limit, so check the reported size before downloading.

In [ ]:
files = sorted(IMAGE_DIR.glob('scene-*.webp'))
if len(files) != 48:
    raise RuntimeError(f'Expected 48 images, found {len(files)}. Generate missing scenes before continuing.')
for number, path in enumerate(files, start=1):
    expected = f'scene-{number:02d}.webp'
    if path.name != expected or path.stat().st_size < 1000:
        raise RuntimeError(f'Invalid or missing image: {expected}')
zip_path = Path('/content/science-channel-48-images.zip')
def build_zip():
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as archive:
        for path in files:
            archive.write(path, arcname=path.name)
build_zip()
size_mb = zip_path.stat().st_size / (1024 * 1024)
if size_mb > 25:
    print(f'ZIP is {size_mb:.1f} MiB; recompressing WebP images to quality 78 to meet the importer limit…')
    for path in files:
        with Image.open(path) as image:
            image.convert('RGB').save(path, 'WEBP', quality=78, method=6)
    build_zip()
    size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f'Validated 48 images. ZIP size: {size_mb:.1f} MiB')
if size_mb > 25:
    raise RuntimeError('The ZIP is still larger than the repository importer limit of 25 MiB. Do not upload it; the images need further compression.')
from google.colab import files
files.download(str(zip_path))

## 5. Send the ZIP to the existing render workflow
1. Download `science-channel-48-images.zip` above.
2. Open your GitHub repository on the `main` branch and upload the ZIP into `science-channel/telegram-uploads/` (create the folder if GitHub does not show it; normally it already exists).
3. Commit the ZIP to `main`.
4. Open **Actions → Telegram ZIP Import → Run workflow**.
5. For `episode_path`, copy the exact episode folder path from the caption of the Telegram prompt document. It looks like `science-channel/daily-episodes/YYYY-MM-DD/episode-001-topic-slug`.
6. For `zip_path`, enter `science-channel/telegram-uploads/science-channel-48-images.zip`.
7. Run the workflow. It validates/imports the images and the existing renderer takes over.

**Do not also send these images individually to the Telegram image receiver.** Use only one intake route for an episode.